# ⚔️ CRUSADER — F04 HELBRECHT
## Remux FFmpeg → final_master.mp4

> *"Helbrecht's will cannot be denied. Where he treads, victory follows."*

---

**Ce notebook tourne sur CPU — GPU non requis.**

### Étapes :
1. Montage Google Drive
2. Vérification FFmpeg
3. Téléchargement des scripts depuis GitHub
4. Configuration des chemins
5. Validation CUSTOS check-out
6. Remux FFmpeg + injection métadonnées
7. Validation CUSTOS check-in
8. Aperçu et téléchargement

---
## Étape 1 — Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive monté sur /content/drive')

---
## Étape 2 — Vérification FFmpeg

> FFmpeg est préinstallé sur Colab. Cette cellule vérifie sa disponibilité.

In [ ]:
import subprocess

result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if result.returncode == 0:
    version_line = result.stdout.splitlines()[0]
    print(f'[OK] {version_line}')
else:
    print('[INSTALL] ffmpeg absent — installation...')
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'], check=True)
    print('[OK] ffmpeg installé.')

---
## Étape 3 — Téléchargement des scripts depuis GitHub

In [ ]:
import urllib.request, os

REPO_RAW    = 'https://raw.githubusercontent.com/kioka8877-ux/CRUSADER/main'
PROJECT_DIR = '/content/crusader'

os.makedirs(PROJECT_DIR, exist_ok=True)

files_to_download = [
    ('F04_HELBRECHT/CODEBASE/crs_f04_helbrecht.py', 'crs_f04_helbrecht.py'),
    ('CRS_CUSTOS.py',                                'CRS_CUSTOS.py'),
]

for repo_path, local_name in files_to_download:
    url      = f'{REPO_RAW}/{repo_path}'
    dst_path = os.path.join(PROJECT_DIR, local_name)
    urllib.request.urlretrieve(url, dst_path)
    size_kb = os.path.getsize(dst_path) / 1024
    print(f'[OK] {local_name} ({size_kb:.1f} KB)')

print()
print('[OK] Tous les scripts téléchargés.')

---
## Étape 4 — Configuration des chemins

> **Modifiez `DRIVE_BASE` si votre structure Google Drive est différente.**

In [ ]:
import os

# ── MODIFIEZ ICI SI NÉCESSAIRE ─────────────────────────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/DRIVE_CRUSADER'
# ──────────────────────────────────────────────────────────────────────────────

PROJECT_DIR = '/content/crusader'
F04_IN      = os.path.join(DRIVE_BASE, 'F04_HELBRECHT', 'IN')
F04_OUT     = os.path.join(DRIVE_BASE, 'F04_HELBRECHT', 'OUT')
os.makedirs(F04_OUT, exist_ok=True)

print('Configuration :')
print(f'  DRIVE_BASE : {DRIVE_BASE}')
print(f'  F04 IN     : {F04_IN}')
print(f'  F04 OUT    : {F04_OUT}')
print()

# Vérification rapide des fichiers d'entrée
for fname in ['short_render.mp4', 'timing.json']:
    path = os.path.join(F04_IN, fname)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f'  [OK] {fname} ({size_mb:.1f} MB)')
    else:
        print(f'  [MANQUANT] {fname} — vérifiez F04/IN/')

---
## Étape 5 — Validation CUSTOS check-out (F04)

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(PROJECT_DIR, 'CRS_CUSTOS.py'),
     '--frigate', 'F04', '--mode', 'check-out', '--drive-base', DRIVE_BASE],
)
if result.returncode != 0:
    print('[STOP] CUSTOS check-out FAIL. Vérifiez short_render.mp4 et timing.json dans F04/IN/.')

---
## Étape 6 — Remux FFmpeg + injection métadonnées

> **Durée estimée : 10-60 secondes selon la taille de la vidéo.**
>
> Opérations effectuées :
> - Copie des flux vidéo et audio (pas de réencodage → rapide)
> - `+faststart` : déplace le MOOV atom en début de fichier (streaming YouTube/TikTok)
> - Injection des métadonnées issues de `timing.json` (titre, date, encodeur)

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(PROJECT_DIR, 'crs_f04_helbrecht.py'),
     '--input',  F04_IN,
     '--output', F04_OUT],
)
if result.returncode != 0:
    print('[STOP] Remux échoué. Consultez les logs ci-dessus.')

---
## Étape 7 — Validation CUSTOS check-in (F04)

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(PROJECT_DIR, 'CRS_CUSTOS.py'),
     '--frigate', 'F04', '--mode', 'check-in', '--drive-base', DRIVE_BASE],
)
if result.returncode != 0:
    print('[STOP] CUSTOS check-in FAIL. final_master.mp4 absent ou trop petit.')
else:
    print('[OK] final_master.mp4 validé — pipeline CRUSADER terminé.')

---
## Étape 8 — Aperçu et téléchargement

In [ ]:
import os
from IPython.display import Video, display

output_path = os.path.join(F04_OUT, 'final_master.mp4')

if os.path.isfile(output_path):
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f'final_master.mp4 — {size_mb:.1f} MB')
    print(f'Chemin : {output_path}')
    print()
    display(Video(output_path, embed=True, width=360))
else:
    print('[ERREUR] final_master.mp4 introuvable.')

In [ ]:
# Téléchargement direct depuis Colab (optionnel)
from google.colab import files
files.download(output_path)